# Phase 0–1: Spark Fundamentals & First Look at the Data
**Apache Spark** is an open-source, unified analytics engine designed for large-scale distributed data processing. It is primarily used to process massive amounts of data in parallel across clusters of computers, making it up to 100 times faster than its predecessor, Hadoop MapReduce, by relying heavily on in-memory data processing rather than slower disk read/write cycles.

**PySpark** is the Python API for Apache Spark, designed for distributed processing of large datasets across multiple machines.

## The project

We're building an end-to-end PySpark pipeline on **NYC Yellow Taxi trip data**
(Jan–Mar 2023, ~9 million trip records) joined with a taxi zone lookup table.
This dataset is genuinely messy (nulls, negative fares, impossible trip
distances, duplicate-ish rows) and big enough that "just use pandas" stops
being a good answer — which is exactly why Spark exists.

## Roadmap

1. **Phase 0–1 (this notebook)** — Spark fundamentals: `SparkSession`, lazy evaluation, partitions, transformations vs. actions. First look at the raw, messy data.
2. **Phase 2 — Cleaning/ETL**: nulls, bad dtypes, dedup, invalid-row filtering.
3. **Phase 3 — Joins**: trips ⋈ zone lookup, broadcast vs. shuffle joins, skew.
4. **Phase 4 — Aggregations & window functions**: busiest zones/hours, running totals, ranking.
5. **Phase 5 — Performance tuning**: shuffle partitions, caching, Adaptive Query Execution (AQE), reading physical plans.
6. **Phase 6 — Delta Lake / Databricks**: ACID tables, time travel, `MERGE`, Unity Catalog concepts.

At the end of each phase: a short recap + a small challenge before moving on.

## Databricks note

Everything here runs in a local Docker container for convenience, but the
*concepts* are identical to a Databricks notebook. The main practical
difference: in Databricks a `SparkSession` (called `spark`) is **already created
for you**, and you get a `%sql` magic cell and Unity Catalog table names instead
of raw file paths. We'll call these out as we go.

## 0 · The *SparkSession* — your handle on the cluster

Everything in Spark starts here. The **`SparkSession`** is your entry point: the
object through which you read data, build queries, and hand work to the cluster.
Think of it as the *remote control* for the engine we talked about — the thing
that talks to the **driver**, which in turn hands tasks to **executors**.

- `master("local[*]")` — run Spark on this one machine, using **all** its cores
  (`*`). This is *local mode*: same API as a real cluster, just no network. Great
  for learning.
- `appName(...)` — a label you'll see in the Spark UI.
- `spark.sql.shuffle.partitions` — remember the default of **200**? Way too many
  for a laptop. We set it low so shuffles don't create hundreds of tiny tasks.

> **Databricks:** skip this whole cell. `spark` already exists the moment your
> notebook attaches to a cluster. Creating your own there is a common beginner
> mistake.

In [0]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("DataForge-Phase0-1")
    .master("local[*]")                          # all cores on this machine
    .config("spark.sql.shuffle.partitions", "8") # sane default for local dev
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")  # quieten the log firehose
spark

## The catch: these three files don't share one schema

The obvious `spark.read.parquet(".../2023-*.parquet")` **fails** here, because the
three monthly files disagree on column *types* — in both directions:

| Column | Some files | Other files |
|---|---|---|
| `VendorID`, `PULocationID`, `DOLocationID` | `int` | `long` |
| `passenger_count`, `RatecodeID` | `bigint` | `double` |

Plus a name-casing clash: `airport_fee` vs `Airport_fee`.

We tried four approaches, each defeated by a *different* `Caused by:` line at the
bottom of the trace (reading that line is the whole debugging skill):

1. **Naive read** → `Parquet column cannot be converted ... Expected: double, Found: INT64`.
2. **`mergeSchema=true`** → `CANNOT_MERGE_INCOMPATIBLE_DATA_TYPE ("BIGINT" and "INT")` — but it usefully printed both schemas.
3. **Explicit `StructType` + `.schema()`** → the vectorized reader still refused the physical `INT64 → double` convert.
4. **Disable the vectorized reader** → `ClassCastException` on a column that clashed the *other* way (double-on-disk vs. declared long). Proof that **no single physical read satisfies all three files**.

**The lesson:** the Parquet reader reads *raw bytes into fixed slots* and refuses
unsafe physical conversions. `.cast()` works on *values* inside the DataFrame
engine and converts safely. So we don't force types at the reader — we **read
each file as-is, cast to one agreed set of types, then `unionByName`** (next
cell). This is exactly how real Bronze ingestion absorbs schema drift.

## 1 · Reading the data (and meeting *lazy evaluation*)

We read three monthly Parquet files at once with a wildcard, plus the small zone
lookup CSV.

Two things worth noticing:

1. **Parquet carries its own schema** (column names + types), so Spark doesn't
   have to guess. CSV doesn't — that's why we pass `inferSchema` for the lookup,
   and why Parquet is the preferred format for big data.
2. **This is lazy.** `spark.read...` returns a DataFrame, but Spark does *not*
   read 9 million rows here. It reads a bit of metadata and records a *plan*. No
   real work happens until you call an **action** (next section).

> **Path note:** Jupyter usually starts in the notebook's own folder
> (`notebooks/`), so the data sits one level up at `../data/raw`. If your
> container mounts things differently, run `os.getcwd()` and adjust.

In [0]:
import os
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

RAW = "../data/raw"   # adjust if your working dir is the project root -> "data/raw"
assert os.path.exists(RAW), f"Can't find {RAW}. Current dir is: {os.getcwd()}"

# The three monthly files disagree on column types (see the note above), so we
# read each file AS-IS and cast to one agreed set of types, then union by name.
TARGET_TYPES = {
    "VendorID": "long",
    "tpep_pickup_datetime": "timestamp",
    "tpep_dropoff_datetime": "timestamp",
    "passenger_count": "double",     # whole number, but double in some files (fix in Silver)
    "trip_distance": "double",
    "RatecodeID": "double",
    "store_and_fwd_flag": "string",
    "PULocationID": "long",
    "DOLocationID": "long",
    "payment_type": "long",
    "fare_amount": "double",
    "extra": "double",
    "mta_tax": "double",
    "tip_amount": "double",
    "tolls_amount": "double",
    "improvement_surcharge": "double",
    "total_amount": "double",
    "congestion_surcharge": "double",
    "airport_fee": "double",
}

def read_and_cast(path: str) -> DataFrame:
    df = spark.read.parquet(path)                    # native types, no forcing at the reader
    df = df.toDF(*[c.lower() for c in df.columns])   # Airport_fee -> airport_fee
    for col, target in TARGET_TYPES.items():
        df = df.withColumn(col, F.col(col.lower()).cast(target))  # value-level cast, safe
    return df.select(*TARGET_TYPES.keys())           # consistent column set + order

paths = [
    f"{RAW}/yellow_tripdata_2023-01.parquet",
    f"{RAW}/yellow_tripdata_2023-02.parquet",
    f"{RAW}/yellow_tripdata_2023-03.parquet",
]

trips = reduce(DataFrame.unionByName, [read_and_cast(p) for p in paths])

zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW}/taxi_zone_lookup.csv")
)

print("DataFrames created (lazily). trips + zones ready.")

## 2 · The one mental model that matters: transformations vs. actions

This is *the* concept to internalise. Every DataFrame method is one of two kinds:

| | **Transformation** | **Action** |
|---|---|---|
| What it does | Describes a *new* DataFrame | Triggers real computation |
| Runs immediately? | **No** — lazy, just records a plan | **Yes** — kicks off a job |
| Returns | Another DataFrame | A value, or a side effect (file, printout) |
| Examples | `select`, `filter`, `withColumn`, `join`, `groupBy`, `orderBy` | `show`, `count`, `collect`, `take`, `write`, `first` |

**Analogy:** transformations are *writing a recipe*; actions are *actually
cooking*. You can stack transformations all day for free — Spark only fires up
the stove when you call an action, and it optimises the whole recipe first.

> **Why it's designed this way:** by seeing the entire plan before running,
> Spark can reorder filters, prune columns, and skip work. Eager execution
> couldn't do that.

**Two beginner traps to avoid:**
- **`.collect()` on a big DataFrame** pulls *every* row to the driver's memory →
  out-of-memory crash. Use `.show(n)` or `.take(n)` to peek.
- **Calling `.count()` over and over** re-runs the whole plan each time (we'll
  fix that with caching in Phase 5). Fine while exploring; costly in a pipeline.

> **Databricks:** `display(df)` is the idiomatic action there — a rich, sortable
> table (and one-click charts). `df.show()` works everywhere.

In [0]:
# printSchema reads only metadata — cheap
trips.printSchema()

In [0]:
# show() IS an action: this is the first time Spark really reads rows
trips.show(5, truncate=False)

In [0]:
# count() is an action too — it scans all three files
print(f"Total trips: {trips.count():,}")
print(f"Zone rows:   {zones.count():,}")

## 3 · Seeing laziness with your own eyes

Run the next two cells and watch the timing. The **transformations** cell returns
instantly — Spark just wrote down the plan. The **action** cell is where the
work (and the wait) actually happens.

In [0]:
# --- pure transformations: returns instantly, computes nothing ---
long_trips = (
    trips
    .select("tpep_pickup_datetime", "trip_distance", "fare_amount", "total_amount")
    .filter(F.col("trip_distance") > 20)   # 20+ mile rides
)
print("Plan recorded. No data read yet.")

In [0]:
# --- an action: NOW Spark reads, filters, and returns rows ---
long_trips.show(5)

## 4 · Partitions — how Spark splits the work

A DataFrame isn't one big blob; Spark chops it into **partitions** — independent
chunks of rows. Each partition is processed by one **task**, on one **core**. So
partitions are the unit of parallelism: with 8 cores and 8 partitions, all 8
work at once; with 1 partition, 7 cores sit idle.

Let's see how many `trips` got. Parquet files usually map to one-or-more
partitions each, so with three monthly files you'll see a small number.

In [0]:
print("Partitions in `trips`:", trips.rdd.getNumPartitions())
# Recall from earlier: repartition(n) reshuffles into n even parts (full shuffle),
# coalesce(n) merges down without a full shuffle. We'll use both in Phase 5.

## 5 · First look at the mess

Now the real reason this dataset is a good teacher: it's dirty. Before we clean
anything in Phase 2, let's *quantify* the problems. Good data engineering starts
with measuring the damage, not guessing.

First, summary stats on the key numeric columns. Watch the `min` values
especially — that's where the nonsense hides.

In [0]:
(trips
 .select("passenger_count", "trip_distance", "fare_amount", "total_amount")
 .summary("count", "min", "25%", "50%", "75%", "max")
 .show())

Now let's count specific data-quality issues in one pass. Each expression counts
rows matching a problem — `F.when(condition, 1)` produces a 1 for bad rows and
null otherwise, and `count()` ignores nulls, so we get a per-problem tally.

In [0]:
issues = trips.select(
    F.count(F.when(F.col("passenger_count").isNull(), 1)).alias("null_passenger"),
    F.count(F.when(F.col("passenger_count") == 0, 1)).alias("zero_passenger"),
    F.count(F.when(F.col("fare_amount") < 0, 1)).alias("negative_fare"),
    F.count(F.when(F.col("trip_distance") == 0, 1)).alias("zero_distance"),
    F.count(F.when(F.col("trip_distance") > 100, 1)).alias("impossible_distance"),
    F.count(F.when(F.col("total_amount") < 0, 1)).alias("negative_total"),
)
issues.show(truncate=False)

And a sanity check on the timestamps — these files are labelled Jan–Mar 2023,
but taxi data is notorious for stray dates and trips that appear to end before
they start.

In [0]:
out_of_range = trips.filter(
    (F.col("tpep_pickup_datetime") < "2023-01-01") |
    (F.col("tpep_pickup_datetime") >= "2023-04-01")
).count()

reversed_time = trips.filter(
    F.col("tpep_dropoff_datetime") < F.col("tpep_pickup_datetime")
).count()

print(f"Pickups outside Jan-Mar 2023: {out_of_range:,}")
print(f"Dropoff before pickup:        {reversed_time:,}")

**Read your own numbers above and sit with them for a second.** You'll typically
find zero-passenger and zero-distance rows in the tens of thousands, a handful of
negative fares, and a few impossible distances. None of it is random — each one
is a decision waiting for Phase 2: *drop it, keep it, or flag it?*

> **Anti-pattern we just committed (on purpose):** we called `.count()` several
> times, and each call re-scanned all three Parquet files from disk. While
> exploring a few times that's fine. In a real pipeline you'd `.cache()` the
> DataFrame first so repeated actions reuse the same in-memory copy — that's a
> Phase 5 topic, but notice the smell now.

## Recap — what you now actually understand

- **`SparkSession`** is your entry point; in Databricks it's pre-made as `spark`.
- **Lazy evaluation:** transformations only *describe* work; nothing runs until an
  action. This is what lets Spark optimise the whole plan.
- **Transformations vs. actions** — the dividing line behind every Spark program.
  (`select`/`filter`/`join` = lazy; `show`/`count`/`collect`/`write` = triggers.)
- **Partitions** are the unit of parallelism: one partition → one task → one core.
- **Parquet** carries schema and is the format of choice; CSV needs `inferSchema`.
- You measured real data-quality problems instead of assuming them.

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

def kind(label, result):
    """A DataFrame back = transformation (lazy). A value back = action (ran a job)."""
    verdict = "transformation" if isinstance(result, DataFrame) else "action"
    print(f"{label:12} -> {verdict:15} (returned {type(result).__name__})")

# --- one of each, run against trips ---
kind("withColumn", trips.withColumn("tip_ratio", F.col("tip_amount") / F.col("fare_amount")))
kind("groupBy",    trips.groupBy("passenger_count").count())   # groupBy+count -> DataFrame
kind("orderBy",    trips.orderBy("fare_amount"))
kind("first",      trips.first())      # returns a Row
kind("take",       trips.take(3))      # returns a list of Rows

## 🎯 Challenge — do these before we start Phase 2

Work through these in new cells below. They're designed to make the concepts
stick, not to trick you.

1. **Predict, then verify.** For each method, say *transformation* or *action*
   before running anything: `withColumn`, `groupBy`, `first`, `write`,
   `orderBy`, `take`. Then test one of each and confirm your call.

2. **Find the outliers.** How many trips have `passenger_count > 6`? Are those
   plausible in a yellow cab, or data errors?

3. **Peek at the plan.** Run `long_trips.explain()`. Find `FileScan` and
   `Filter` in the output. Notice there's no separate "read" step until an
   action fires — the scan *is* the read. (Look for `PushedFilters` too — Spark
   pushes your filter down into the file read. Why is that a good thing?)

4. **Combine the damage.** Write a single filter that flags a row if it hits *any*
   of the issues from Section 5, then compute what **percentage** of all trips
   are affected. (Hint: build one big boolean `Column` with `|`, filter, count,
   divide by the total.)

5. **Databricks translation.** How would the Section 0 setup cell change if you
   were running this in a Databricks notebook against a Unity Catalog table
   called `nyc.taxi.yellow_trips`? (Two changes: the session, and how you read.)

Post your answers (or the notebook) and I'll review them — then we move on to
**Phase 2: Cleaning & ETL**.

## Further reading

- [PySpark Getting Started](https://spark.apache.org/docs/latest/api/python/getting_started/index.html) — installation + the "Quickstart: DataFrame" page, good next read after this notebook.
- [PySpark API Reference](https://spark.apache.org/docs/latest/api/python/reference/index.html) — look up any `DataFrame`/`pyspark.sql.functions` method you see used.
- [RDD Programming Guide](https://spark.apache.org/docs/latest/rdd-programming-guide.html) — what DataFrames are built on under the hood; not needed day-to-day, but useful once for context on partitions/lazy evaluation.
- [Spark SQL, DataFrames and Datasets Guide](https://spark.apache.org/docs/latest/sql-programming-guide.html) — the conceptual guide covering reading data sources (Parquet vs. CSV), joins, and more.

> **Version note:** these links point at `latest` (Spark 4.2.0 at time of writing).
> Our Docker image runs **Spark 3.5.3** — if the current docs mention
> something that doesn't match what you see here, use the version selector on
> the docs page to switch to 3.5.x before assuming it's a bug.
